# Phase 1 · Lesson 6: Reusable Python and the scikit-learn Estimator API
**ML Engineer Journey · Benson M. Merita**

**Goal:** go from "code in a notebook" to **software an ML team would accept**. Your scratch algorithms from Lessons 1–2 now live in a real Python package, **`mlscratch/`**, at the root of your repo:

```
MachineLearning/
├── mlscratch/              ← the package (importable code)
│   ├── __init__.py         ← public API + version
│   ├── preprocessing.py    ← StandardScalerScratch
│   ├── linear_model.py     ← LinearRegressionScratch (normal equations + gradient descent)
│   ├── decomposition.py    ← PCAScratch (SVD)
│   └── metrics.py          ← mean_squared_error
├── tests/
│   └── test_mlscratch.py   ← 12 automated tests (pytest)
└── pyproject.toml          ← makes it installable:  pip install -e .
```

Every class follows the **scikit-learn estimator contract** and passes scikit-learn's own compatibility suite (`check_estimator`). So your from-scratch code works inside `Pipeline`, `ColumnTransformer`, `cross_val_score` and `GridSearchCV`, unchanged.

| Section | What you get |
|---|---|
| 1 · Intuition | Why notebooks aren't enough |
| 2 · The math | Estimators as maps, pipelines as composition, why GD converges iff $\eta < 2/\lambda_{\max}$ |
| 3 · From scratch | Read the package: the API rules, line by line; a gradient-descent solver |
| 4 · With libraries | Your classes inside `Pipeline` / `ColumnTransformer` / `GridSearchCV`; `pytest`; `check_estimator` |
| 5 · Dataset | Diamonds (54k rows, numeric + categorical) |
| 6–8 | Exercises, quiz, mistakes & interview questions |
| 🎬 Demo | Step-size race GIF, pipeline diagram, "all checks passed" |

---
## 1 · Intuition

Notebooks are great for exploring, but they're a poor home for code you rely on. You copy a function into the next notebook, fix a bug in one copy but not the other, and three weeks later nobody knows which version produced the result.

Professional ML code is organised like a **kitchen with standard equipment**. Every appliance has the same buttons:
- **`fit`**: learn from data (the oven learns the recipe),
- **`transform`** or **`predict`**: apply what was learned to new data,
- **`get_params` / `set_params`**: read and change the settings.

Because every tool has the same buttons, you can **chain** them (scale → compress → regress) and **automate** them (try 20 settings, cross-validate each) without writing special code for each one. That shared interface is scikit-learn's estimator API. When your own classes follow it, they plug into the whole ecosystem.

**Tests** are the kitchen's safety inspection: small automated checks that prove the code still does what it promised every time you change it.

**Real ML engineering connection:** production ML is mostly this. You'll find importable packages, consistent interfaces, tests run automatically on every push (CI), and pipelines that bundle preprocessing with the model, so the *exact same* transformations run in training and in production (Phase 7).

---
## 2 · The math

**An estimator is a map from data to parameters.** `fit` computes $\hat\theta = \mathcal{A}(X_{\text{train}}, y_{\text{train}})$, for example $\hat\theta = (\mu, \sigma)$ for a scaler or $\hat\theta = (w, b)$ for regression. `transform`/`predict` evaluate a function with those **frozen** parameters: $f_{\hat\theta}(x)$.

**A pipeline is function composition.** For transformers $g_1, g_2$ and a model $h$:
$$\text{predict}(x) = h_{\hat\theta_3}\big(g_{2,\hat\theta_2}(g_{1,\hat\theta_1}(x))\big)$$
with the parameters fitted **sequentially on training data only**: $\hat\theta_1$ from $X$, then $\hat\theta_2$ from $g_1(X)$, then $\hat\theta_3$ from $g_2(g_1(X))$. Inside cross-validation the whole pipeline is re-fitted on each training fold, so **no statistic from the validation fold leaks in**. That's the reason to put scaling *inside* the pipeline.

**Why gradient descent needs $\eta < 2/\lambda_{\max}$.** The MSE $L(w) = \frac1n\lVert Aw - y\rVert^2$ is quadratic with Hessian $H = \frac2n A^\top A$ (symmetric PSD, eigenvalues $0 \le \lambda_{\min} \le \dots \le \lambda_{\max}$). Let $e_k = w_k - w^*$. Since $\nabla L(w_k) = H e_k$:
$$e_{k+1} = e_k - \eta H e_k = (I - \eta H)\, e_k \quad\Rightarrow\quad e_k = (I - \eta H)^k e_0$$
In the eigenbasis of $H$, each component is multiplied by $(1 - \eta\lambda_i)$ every step. It shrinks iff $|1 - \eta\lambda_i| < 1 \iff 0 < \eta < 2/\lambda_i$. The binding constraint is the largest eigenvalue:
$$\boxed{0 < \eta < \frac{2}{\lambda_{\max}}}$$
The default $\eta = 1/\lambda_{\max}$ is safe. The **convergence rate** is limited by the slowest direction, $1 - \eta\lambda_{\min} = 1 - \lambda_{\min}/\lambda_{\max} = 1 - 1/\kappa(H)$. So **ill-conditioned features (large $\kappa$) make gradient descent slow**: it needs about $\kappa \log(1/\varepsilon)$ steps. That's Lesson 2's condition number deciding training speed, and the reason we standardise features before gradient-based training.

---
## 3 · From scratch: read your package
### 3.1 Import it
Best practice is to install it once in editable mode from the repo root (`pip install -e .`), after which `import mlscratch` works from anywhere and picks up your edits. This cell also works without installing, by finding the repo root.

In [1]:
import sys, inspect
from pathlib import Path
import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import mlscratch
from mlscratch import StandardScalerScratch, LinearRegressionScratch, PCAScratch, mean_squared_error
print("mlscratch", mlscratch.__version__, "from", Path(mlscratch.__file__).parent.name + "/")
print("public API:", mlscratch.__all__)

mlscratch 0.1.0 from mlscratch/
public API: ['StandardScalerScratch', 'LinearRegressionScratch', 'PCAScratch', 'mean_squared_error']


### 3.2 The five API rules, shown in your own code
**Rule 1: `__init__` only stores hyperparameters, unchanged.** No validation, no computation, no data. That lets scikit-learn `clone` an estimator and read its settings with `get_params()`.

In [2]:
print(inspect.getsource(LinearRegressionScratch.__init__))
m = LinearRegressionScratch(solver="gd", lr=0.05)
print("get_params():", m.get_params())

    def __init__(self, solver="normal", lr=None, max_iter=5000, tol=1e-10, fit_intercept=True):
        self.solver = solver
        self.lr = lr
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept

get_params(): {'fit_intercept': True, 'lr': 0.05, 'max_iter': 5000, 'solver': 'gd', 'tol': 1e-10}


**Rule 2: `fit` validates input, learns attributes ending in `_`, and returns `self`.**

In [3]:
print(inspect.getsource(StandardScalerScratch.fit))

    def fit(self, X, y=None):
        X = validate_data(self, X, dtype=float)            # checks X, records n_features_in_
        self.mean_ = X.mean(axis=0) if self.with_mean else np.zeros(X.shape[1])
        std = X.std(axis=0) if self.with_std else np.ones(X.shape[1])
        self.scale_ = np.where(std == 0, 1.0, std)         # constant column: avoid division by zero
        return self                                        # returning self allows chaining



**Rule 3: `transform` / `predict` check the model is fitted and that the input has the same number of features.** `validate_data(..., reset=False)` does the feature check.

In [4]:
print(inspect.getsource(LinearRegressionScratch.predict))

    def predict(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, dtype=float, reset=False)
        return X @ self.coef_ + self.intercept_



**Intentional error: predicting before fitting.**

In [5]:
LinearRegressionScratch().predict(np.zeros((2, 3)))

NotFittedError: This LinearRegressionScratch instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

*`NotFittedError`, a clear message rather than a confusing `AttributeError: coef_`. Good libraries fail **early and loudly**.*

**Rule 4: inherit the mixins.** `TransformerMixin` gives you `fit_transform` for free. `RegressorMixin` gives you `.score()` (R²). `BaseEstimator` gives you `get_params`/`set_params`/`clone` and a nice display.
**Rule 5: docstrings.** Every class documents its parameters and learned attributes (`help(LinearRegressionScratch)`).

### 3.3 The gradient-descent solver (Lesson 1's gradient, now with a guaranteed step size)

In [6]:
print(inspect.getsource(LinearRegressionScratch._gradient_descent))

    def _gradient_descent(self, A, y):
        n = len(A)
        L = 2.0 / n * np.linalg.norm(A, 2) ** 2            # largest eigenvalue of the Hessian (2/n) A^T A
        lr = self.lr if self.lr is not None else (1.0 / L if L > 0 else 1.0)
        w = np.zeros(A.shape[1])
        history = []
        with np.errstate(over="ignore", invalid="ignore"):  # divergence is detected and reported below
            for it in range(self.max_iter):
                r = A @ w - y
                loss = r @ r / n
                history.append(float(loss))
                if not np.isfinite(loss):
                    raise FloatingPointError(
                        f"gradient descent diverged at iteration {it}: lr={lr:.3g} exceeds 2/L={2 / L:.3g}")
                if it > 0 and abs(history[-2] - loss) <= self.tol * max(history[-2], 1e-300):
                    break
                w = w - lr * (2.0 / n) * (A.T @ r)          # w <- w - lr * gradient
        return w, history, it + 1, lr



*`L` is $\lambda_{\max}$ of the Hessian, computed as $\frac2n\lVert A\rVert_2^2$ (the squared largest singular value). It never forms $A^\top A$. The loop stops on relative improvement < `tol` and **raises** if the loss blows up, so failure is never silent.*

---
## 5 · Practice dataset
**Diamonds** (via seaborn; about 54,000 round-cut diamonds)
- **Numeric:** `carat`, `depth` (%), `table` (%), `x`, `y`, `z` (dimensions in mm).
- **Categorical (ordered):** `cut` (Fair → Ideal), `color` (J worst → D best), `clarity` (I1 → IF).
- **Target:** `price` (US$). It's heavily right-skewed, so we model **log(price)** (Lesson 4's distribution habit).
- **Task:** regression. Why it's good here: mixed column types need a `ColumnTransformer`, it's big enough for timing to matter, and $x, y, z$, carat are highly collinear (large κ, slow gradient descent).
- **Load:** `sns.load_dataset("diamonds")`.

In [7]:
import pandas as pd, seaborn as sns
dia = sns.load_dataset("diamonds")
dia = dia[(dia[["x", "y", "z"]] > 0).all(axis=1)].copy()         # 20 rows have impossible 0 mm dimensions
num_cols = ["carat", "depth", "table", "x", "y", "z"]
cat_cols = ["cut", "color", "clarity"]
X_df = dia[num_cols + cat_cols]
y = np.log(dia["price"].to_numpy())
print("rows:", len(dia), "| price skew:", round(dia["price"].skew(), 2), "-> log(price) skew:", round(pd.Series(y).skew(), 2))
Z_num = StandardScalerScratch().fit_transform(dia[num_cols])
print("condition number of scaled numeric features:", round(np.linalg.cond(Z_num), 1))

rows: 53920 | price skew: 1.62 -> log(price) skew: 0.12
condition number of scaled numeric features: 18.4


*A condition number of about 18 comes from carat and x, y, z, which all measure size. The Hessian of the MSE squares it: κ(H) = κ(Z)² ≈ 340. By section 2's maths, gradient descent will crawl along the poorly scaled direction. Section 4.4 measures it.*

---
## 4 · Your classes inside scikit-learn's machinery
### 4.1 `ColumnTransformer` + `Pipeline` with *your* scaler and *your* model

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold

X_tr, X_te, y_tr, y_te = train_test_split(X_df, y, test_size=0.2, random_state=42)

def make_pipe(scaler, model):
    pre = ColumnTransformer([("num", scaler, num_cols),
                             ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)])
    return Pipeline([("prep", pre), ("model", model)])

ours = make_pipe(StandardScalerScratch(), LinearRegressionScratch()).fit(X_tr, y_tr)
ref  = make_pipe(StandardScaler(), LinearRegression()).fit(X_tr, y_tr)

pred_o, pred_r = ours.predict(X_te), ref.predict(X_te)
print(f"test R^2   ours: {ours.score(X_te, y_te):.5f} | sklearn: {ref.score(X_te, y_te):.5f}")
print(f"test MSE   ours: {mean_squared_error(y_te, pred_o):.6f} (log-price units)")
print("predictions identical:", np.allclose(pred_o, pred_r))
print(f"typical error: about {100 * (np.exp(np.sqrt(mean_squared_error(y_te, pred_o))) - 1):.0f}% of price")

test R^2   ours: 0.95367 | sklearn: 0.95367
test MSE   ours: 0.047665 (log-price units)
predictions identical: True
typical error: about 24% of price


*Same predictions to machine precision. Your hand-written classes are interchangeable with scikit-learn's. `.score` (R²) came free from `RegressorMixin`.*

### 4.2 Cross-validation: the pipeline is re-fitted on every fold (no leakage)

In [9]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(make_pipe(StandardScalerScratch(), LinearRegressionScratch()), X_df, y, cv=cv, scoring="r2")
print("5-fold R^2:", scores.round(4), "| mean", scores.mean().round(4), "+/-", scores.std().round(4))

5-fold R^2: [0.9803 0.9816 0.9808 0.9534 0.9816] | mean 0.9755 +/- 0.0111


*`cross_val_score` calls `clone()` on your pipeline for each fold. That works only because `__init__` follows Rule 1.*

### 4.3 Grid search over a hyperparameter of *your* PCA

In [10]:
from sklearn.model_selection import GridSearchCV
pca_pipe = Pipeline([("scale", StandardScalerScratch()), ("pca", PCAScratch()), ("model", LinearRegressionScratch())])
grid = GridSearchCV(pca_pipe, {"pca__n_components": [1, 2, 3, 4, 5, 6]}, cv=cv, scoring="r2")
grid.fit(dia[num_cols], y)
res = pd.DataFrame(grid.cv_results_)[["param_pca__n_components", "mean_test_score"]]
print(res.round(4).to_string(index=False))
print("best:", grid.best_params_)

 param_pca__n_components  mean_test_score
                       1           0.9018
                       2           0.9036
                       3           0.9074
                       4           0.9129
                       5           0.8914
                       6           0.9247
best: {'pca__n_components': 6}


*One principal component already reaches R² ≈ 0.90, and all six only about 0.92. The six numeric features are mostly **one direction**, "how big is the diamond", which is the collinearity the condition number flagged. Using `step__param` naming, `GridSearchCV` reached inside the pipeline and set a parameter on your class through `set_params`.*

*Why isn't the score monotonic in k (k = 5 dips)? Adding a component can't lower the **training** fit, but cross-validated scores can move around because of a few **impossible rows**: some diamonds have y or z of 30–60 mm (a 2-carat stone listed as 58.9 mm wide!). When such a row lands in a validation fold, its prediction is wild. Detecting these is Phase 2 (outliers). EDA would have caught them first.*

### 4.4 Measuring the convergence theory: iterations vs condition number

In [11]:
import time
A_num = np.c_[np.ones(len(Z_num)), Z_num]
H_eigs = np.linalg.eigvalsh(2 / len(A_num) * A_num.T @ A_num)
print(f"Hessian: lambda_max = {H_eigs[-1]:.2f}, lambda_min = {H_eigs[0]:.4f}, kappa = {H_eigs[-1] / H_eigs[0]:,.0f}")

rows = []
for name, cols in [("well-conditioned: carat, depth, table", ["carat", "depth", "table"]),
                   ("collinear: all six", num_cols)]:
    Xs = StandardScalerScratch().fit_transform(dia[cols])
    A = np.c_[np.ones(len(Xs)), Xs]
    e = np.linalg.eigvalsh(2 / len(A) * A.T @ A)
    t0 = time.perf_counter()
    m = LinearRegressionScratch(solver="gd", max_iter=50_000, tol=1e-12).fit(Xs, y)
    exact = LinearRegressionScratch().fit(Xs, y)
    rows.append({"features": name, "kappa(H)": round(e[-1] / e[0]), "GD iterations": m.n_iter_,
                 "seconds": round(time.perf_counter() - t0, 2),
                 "max |w_gd - w_exact|": f"{np.abs(m.coef_ - exact.coef_).max():.1e}"})
pd.DataFrame(rows)

Hessian: lambda_max = 7.89, lambda_min = 0.0232, kappa = 340


,features,kappa(H),GD iterations,seconds,max |w_gd - w_exact|
0,"well-conditioned: carat, depth, table",2,23,0.01,1.6e-07
1,collinear: all six,340,3615,1.27,2.7e-05


*The theory holds up: as κ grows, gradient descent needs **far** more iterations to reach the same answer. Better scaling, decorrelating (PCA) or smarter optimisers (momentum, Adam in Phase 6) all attack this same quantity.*

### 4.5 Automated tests with `pytest`

In [12]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "--color=no", "-p", "no:cacheprovider"],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-600:])

............                                                             [100%]
12 passed in 0.89s



*12 tests check behaviour against scikit-learn, edge cases (constant columns, wrong feature counts, unfitted models) and a divergence test that proves a too-large step raises. Run `python -m pytest` from the repo root before every commit. Exercise 5 makes GitHub run them automatically.*

### 4.6 The official compatibility exam: `check_estimator`

In [13]:
import warnings
from sklearn.utils.estimator_checks import check_estimator
for est in [StandardScalerScratch(), LinearRegressionScratch(), LinearRegressionScratch(solver="gd"), PCAScratch()]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        check_estimator(est)
    print(f"PASSED  {est!r}")

PASSED  StandardScalerScratch()


PASSED  LinearRegressionScratch()


PASSED  LinearRegressionScratch(solver='gd')
PASSED  PCAScratch()


*scikit-learn runs dozens of checks: cloning, pickling, input validation, NaN handling, idempotent fitting, feature-count consistency and more. Passing all of them is a concrete, verifiable claim for your CV: "my estimators pass scikit-learn's `check_estimator`".*

| You wrote | You got for free from scikit-learn |
|---|---|
| `__init__` that stores params | `get_params`, `set_params`, `clone`, `GridSearchCV` support |
| `fit` + `transform` + `TransformerMixin` | `fit_transform`, use in `Pipeline` / `ColumnTransformer` |
| `fit` + `predict` + `RegressorMixin` | `.score()` (R²), `cross_val_score` |
| `validate_data` + `check_is_fitted` | consistent errors, feature-count checks |

---
## 🎬 Demo visuals for your post
### Demo A: the step-size race (`assets/step_size_race.gif`)

In [14]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
Xs = StandardScalerScratch().fit_transform(dia[["carat", "depth", "table"]].to_numpy())
A = np.c_[np.ones(len(Xs)), Xs]; n = len(A)
L = 2 / n * np.linalg.norm(A, 2) ** 2
w_star, *_ = np.linalg.lstsq(A, y, rcond=None)
loss_star = np.mean((A @ w_star - y) ** 2)
steps = 60
def run(lr):
    w, out = np.zeros(A.shape[1]), []
    for _ in range(steps):
        r = A @ w - y; out.append(np.mean(r ** 2) - loss_star); w = w - lr * 2 / n * A.T @ r
    return np.array(out)
runs = {"lr = 0.3/L  (too timid)": (run(0.3 / L), "#8ab17d"),
        "lr = 1/L    (default)": (run(1.0 / L), "#2a9d8f"),
        "lr = 1.9/L  (zig-zag)": (run(1.9 / L), "#e9c46a"),
        "lr = 2.05/L (diverges)": (run(2.05 / L), "#c44536")}
fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=100)
def update(k):
    ax.clear()
    for lab, (curve, col) in runs.items():
        ax.plot(range(k + 1), np.maximum(curve[: k + 1], 1e-12), color=col, lw=2.2, label=lab)
    ax.set_yscale("log"); ax.set_xlim(0, steps); ax.set_ylim(1e-9, 1e3)
    ax.set(xlabel="iteration", ylabel="loss - optimal loss (log)")
    ax.set_title(f"Gradient descent converges iff lr < 2/lambda_max   (step {k})", loc="left", fontweight="bold")
    ax.legend(frameon=False, loc="lower left", fontsize=9); ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
anim = FuncAnimation(fig, update, frames=list(range(steps)) + [steps - 1] * 15)
anim.save("assets/step_size_race.gif", writer=PillowWriter(fps=12))
update(steps - 1); fig.savefig("assets/step_size_race.png", bbox_inches="tight"); plt.close(fig)
print("Saved assets/step_size_race.gif / .png")

Saved assets/step_size_race.gif / .png


### Demo B: the pipeline diagram (`assets/pipeline_diagram.html`). Open it in a browser and click the boxes.

In [15]:
from sklearn.utils import estimator_html_repr
html = estimator_html_repr(make_pipe(StandardScalerScratch(), LinearRegressionScratch(solver="gd")))
Path("assets/pipeline_diagram.html").write_text("<h3>Pipeline built from my own estimators</h3>" + html, encoding="utf-8")
print("Saved assets/pipeline_diagram.html")
make_pipe(StandardScalerScratch(), LinearRegressionScratch(solver="gd"))

Saved assets/pipeline_diagram.html


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

---
## 6 · Exercises (answers hidden; commit on a branch and open a PR, as in Lesson 5)
1. **(Easy) Metrics module:** Add `r2_score` and `mean_absolute_error` to `mlscratch/metrics.py`, export them in `__init__.py`, and add tests comparing them with `sklearn.metrics`.
2. **(Easy–Medium) `MinMaxScalerScratch`:** Add it to `preprocessing.py` (parameter `feature_range=(0, 1)`), with tests, and make it pass `check_estimator`.
3. **(Medium) Ridge regression:** Add `alpha=0.0` to `LinearRegressionScratch`. Normal solver: solve $(A^\top A + \alpha D)w = A^\top y$ where $D = \operatorname{diag}(0, 1, \dots, 1)$, so the **intercept isn't penalised**. GD solver: add the penalty to the gradient. Test against `sklearn.linear_model.Ridge`. *(Bonus: show that ridge reduces κ, and so speeds up gradient descent.)*
4. **(Medium–Hard) Mini-batch SGD:** Add `solver="sgd"` with `batch_size` and `random_state` parameters (use `default_rng`, Lesson 2). Plot loss against epoch for batch sizes 1, 32 and full-batch on diamonds. Discuss noise vs speed.
5. **(Challenge) Continuous integration:** Create `.github/workflows/tests.yml` so GitHub runs `pip install -e . pytest && python -m pytest` on every push and pull request. Add the passing-tests badge to your README.

## 7 · Mini-quiz
1. Why must `__init__` not validate or modify its parameters? Which scikit-learn feature breaks if it does?
2. Why put `StandardScaler` *inside* the pipeline instead of scaling `X` once before `cross_val_score`?
3. With $\lambda_{\max} = 4$, which of these step sizes converge for MSE: 0.1, 0.45, 0.5, 0.6? Why?
4. What do the trailing underscores in `coef_`, `mean_`, `n_features_in_` signal?

## 8 · Common mistakes & interview questions
**Common mistakes**
- **Logic in `__init__`** (validation, conversions, data): `clone` and `GridSearchCV` silently use stale values.
- **Forgetting `return self`** in `fit`: breaks chaining and `fit_transform`.
- **Fitting preprocessing on all the data** instead of inside the pipeline: leakage that inflates CV scores.
- **Copy-pasted notebook code** instead of a shared module, so fixes don't propagate.
- **No tests, or tests that only check "it runs"**: compare against a trusted reference and include edge cases.
- **Silent failures:** gradient descent quietly returning NaN weights. Detect it and raise.
- **Hard-coded paths** like `C:\\Users\\...` in package code: use `pathlib` and parameters.

**Interview questions**
1. Explain the scikit-learn estimator API. How would you write a custom transformer?
2. What is data leakage, and how do pipelines prevent it?
3. Why does feature scaling matter for gradient descent but not for the normal equations?
4. Derive the maximum stable learning rate for gradient descent on a quadratic.
5. How do you structure and test an ML codebase? What would you put in CI?
6. Normal equations vs gradient descent: when would you choose each?

---
### ✅ Lesson 6 checklist
- [ ] `pip install -e .` from the repo root, then `python -m pytest`: 12 passed
- [ ] Notebook runs top to bottom
- [ ] Exercises 1–5 attempted (on a branch, merged via PR)
- [ ] Quiz answered
- [ ] Post: step-size GIF or "my estimators pass check_estimator" (see `SOCIAL_PACK.md`)

**Next:** 🏁 **Phase 1 portfolio project**, an end-to-end EDA + modelling report on a real Kenyan/African open dataset, using everything from Lessons 1–6.